# EquipED DPO Training Template

Before running: open the EquipED admin panel, go to **Training Data**
for the agent you're training, click **Start Training Job**, and paste
the resulting `download_url` and `upload_url` below.

**The download URL is single-use.** If this notebook disconnects or you
need to re-run the fetch cell, re-running it will fail with a 404 --
the token is already spent. Go back to the admin panel and start a new
training job to get a fresh pair of URLs; there is no way to reuse or
extend the old ones.

This notebook does not include the actual LoRA/PEFT training loop --
fill that in under the `# TODO` cell based on your chosen base model
and hyperparameters.

In [ ]:
DOWNLOAD_URL = "PASTE_DOWNLOAD_URL_HERE"
UPLOAD_URL = "PASTE_UPLOAD_URL_HERE"

In [ ]:
import io
import json
import zipfile

import requests

response = requests.get(DOWNLOAD_URL)
response.raise_for_status()

with zipfile.ZipFile(io.BytesIO(response.content)) as zf:
    pairs = [json.loads(line) for line in zf.read("pairs.jsonl").decode("utf-8").splitlines() if line]
    manifest = json.loads(zf.read("manifest.json").decode("utf-8"))

print(f"Loaded {len(pairs)} DPO pairs for agent={manifest['agent_id']}")

## Training

The cells below install dependencies, build a train/validation split from
the fetched `pairs`, load a 4-bit quantized base model with a LoRA
adapter, run TRL's `DPOTrainer`, print a sanity-check evaluation (this is
NOT the project's held-out test set -- see the printed note), and bundle a
small provenance manifest into the adapter before the push-back cell zips
and uploads it.

The LoRA rank/alpha and DPO hyperparameters below are a reasonable
starting recipe for a small model on Colab's free-tier GPU, not a tuned
result -- edit them directly in the cells if you have reason to.

In [ ]:
!pip install -q unsloth trl peft bitsandbytes datasets

In [ ]:
from datasets import Dataset

dataset = Dataset.from_list(pairs)

MIN_PAIRS_FOR_EVAL_SPLIT = 20
if len(pairs) >= MIN_PAIRS_FOR_EVAL_SPLIT:
    split = dataset.train_test_split(test_size=0.1, seed=42)
    train_dataset = split["train"]
    eval_dataset = split["test"]
else:
    train_dataset = dataset
    eval_dataset = None
    print(
        f"Only {len(pairs)} pairs available (< {MIN_PAIRS_FOR_EVAL_SPLIT}) "
        "-- skipping the held-out validation split and training on all of "
        "them. The eval/sanity-check cell below will be skipped too."
    )

print(f"train_dataset: {len(train_dataset)} pairs")
if eval_dataset is not None:
    print(f"eval_dataset: {len(eval_dataset)} pairs")

In [ ]:
from unsloth import FastLanguageModel

MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/gemma-3-4b-it",
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.0,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

In [ ]:
from trl import DPOConfig, DPOTrainer
from unsloth import is_bfloat16_supported

ADAPTER_DIR = "./trained_adapter"

training_args = DPOConfig(
    output_dir=ADAPTER_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=5e-6,
    num_train_epochs=1,
    beta=0.1,
    max_length=MAX_SEQ_LENGTH,
    max_prompt_length=1536,
    eval_strategy="steps" if eval_dataset is not None else "no",
    eval_steps=20,
    logging_steps=5,
    save_strategy="no",
    report_to="none",
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
)
trainer = DPOTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
)
trainer.train()

model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

In [ ]:
if eval_dataset is not None:
    try:
        metrics = trainer.evaluate()
        print(f"eval_loss: {metrics['eval_loss']:.4f}")
        print(
            "reward accuracy (chosen > rejected): "
            f"{metrics.get('eval_rewards/accuracies', 'n/a')}"
        )
        print(
            "NOTE: this is an in-run sanity check on a random 10% split of "
            "THIS training run's own data -- it is not the project's "
            "held-out test set, and does not compare against the base model. "
            "A low accuracy here is a strong signal something went wrong; a "
            "high accuracy is not by itself a green light to deploy."
        )
    except Exception as exc:
        # This cell is a soft, non-blocking sanity check -- a crash here
        # (e.g. a known transformers/notebook-progress-callback quirk on
        # very short training runs) must never stop the notebook from
        # reaching the push-back cell below.
        metrics = None
        print(f"Eval sanity check failed to run ({exc!r}); skipping it. "
              "This does not affect the trained adapter -- continuing to "
              "the upload cell.")
else:
    metrics = None
    print("Skipped eval (too few pairs for a meaningful held-out split).")

In [ ]:
import json as _json
import os

training_manifest = {
    "source_job_manifest": manifest,
    "base_model": "unsloth/gemma-3-4b-it",
    "max_seq_length": MAX_SEQ_LENGTH,
    "lora_config": {
        k: v
        for k, v in model.peft_config["default"].to_dict().items()
        if k in ("r", "lora_alpha", "lora_dropout", "target_modules")
    },
    "training_args": {
        "learning_rate": training_args.learning_rate,
        "num_train_epochs": training_args.num_train_epochs,
        "beta": training_args.beta,
        "per_device_train_batch_size": training_args.per_device_train_batch_size,
        "gradient_accumulation_steps": training_args.gradient_accumulation_steps,
    },
    "pair_count": len(pairs),
    "eval_metrics": metrics,
}
training_manifest["lora_config"]["target_modules"] = sorted(
    training_manifest["lora_config"]["target_modules"]
)
with open(os.path.join(ADAPTER_DIR, "training_manifest.json"), "w") as f:
    _json.dump(training_manifest, f, indent=2)

print(f"Wrote {ADAPTER_DIR}/training_manifest.json")

In [ ]:
import os
import zipfile as zf_module

ADAPTER_ZIP_PATH = "trained_adapter.zip"
with zf_module.ZipFile(ADAPTER_ZIP_PATH, mode="w", compression=zf_module.ZIP_DEFLATED) as zf:
    for root, _dirs, files in os.walk(ADAPTER_DIR):
        for name in files:
            full_path = os.path.join(root, name)
            zf.write(full_path, arcname=name)

with open(ADAPTER_ZIP_PATH, "rb") as f:
    upload_response = requests.post(
        UPLOAD_URL,
        files={"file": ("adapter.zip", f, "application/zip")},
    )
upload_response.raise_for_status()
print("Adapter uploaded:", upload_response.json())